In [2]:
# 01 导入所需的包
import json
import re
from tqdm import tqdm
import time
import numpy as np
import pandas as pd
from glove import Corpus,Glove
from flashtext import KeywordProcessor
from collections import Counter

In [3]:
# 02 对json格式的源数据做调整
data = json.load(open(f'datas\initial_data\wos_LIS.json','r'))
# 去除不对的DOI符号
for tmp in tqdm(data):
    ref_list = [ref.strip(']') for ref in tmp['CI']]
    tmp['CI'] = ref_list

100%|██████████| 57722/57722 [00:00<00:00, 199992.74it/s]


In [4]:
data_df = pd.DataFrame(data)
data_df

,DOI,KW,AB,TI,YE,CI
0,10.1287/isre.1100.0313,[HistoryOfIsr],,the early year of iisri recollection of the ed...,2010,"[10.1145/63238.63241, 10.1145/944217.944221, 1..."
1,10.1287/isre.1100.0315,"[SocialMedium, SocialCommerce, SponsoredSearch...",SponsoredSearch is the mechanism whereby adver...,SponsoredSearch and MarketEfficiency,2010,"[10.1016/j.jfineco.2004.06.007, 10.1257/aer.99..."
2,10.1287/isre.1080.0218,"[CollaborativeOnlineShopping, SharedNavigation...",prior study investigating businesstoconsumer e...,let shop online together an empirical investig...,2010,"[10.1287/isre.13.4.404.72, 10.1016/S1567-4223(..."
3,10.1287/isre.1080.0217,"[KnowledgeManagement, Codification, Knowledge-...",current KnowledgeManagement km technology and ...,the interaction between knowledge Codification...,2010,"[10.2307/3250961, 10.1287/mnsc.49.4.432.14428,..."
4,10.2753/MIS0742-1222270302,"[BusinessProcessOutsourcing, Offshoring, Organ...",this paper identifies and analyzes firmlevel c...,OrganizationalLearning and capability for onsh...,2010,"[10.1002/smj.4250130503, 10.2307/249554, 10.12..."
...,...,...,...,...,...,...
57717,10.17705/1jais.00775,"[InformationPrivacy, PrivacyAAState, PrivacyBo...",to reconcile the personalizationprivacy parado...,reconciling the personalizationprivacy paradox...,2023,"[10.5465/amd.2014.0018, 10.1177/10944281145479..."
57718,10.1177/10497323221135974,"[RareDisease, RareDisorder, Disability, Advoca...",in a twostudy project researcher used qualitat...,if not me who Awareness and selfadvocacyrelate...,2023,"[10.1080/13603116.2021.1888323, 10.1111/jopy.1..."
57719,10.1093/jcmc/zmac027,"[SocialPresence, Copresence, Explication, Huma...",initially the province of telecommunication an...,capturing SocialPresence concept Explication t...,2023,"[10.2307/2786027, 10.1162/105474603322761270, ..."
57720,10.1093/jcmc/zmac024,"[VisualMisinformation, Out-of-contextVisualMis...",a a significant source of misinformation outof...,fighting cheapfakes using a DigitalMediumLiter...,2023,"[10.1080/23299460.2018.1495026, 10.1177/205395..."


In [4]:
# 02 对关键词进行提取，看看是否存在于主题和摘要中
data_df = pd.DataFrame(data)

def process_keywords(row):
    # 创建关键词处理器
    keyword_processor = KeywordProcessor()
    kw_list = row['KW']
    for kw in kw_list:
        keyword_processor.add_keyword(kw) 
    cleaned_abstract = re.sub(r'[^\w\s]', ' ', row['AB'])  # 将摘要中的标点符号替换为空格
    match_keywords = keyword_processor.extract_keywords(cleaned_abstract)  # 使用关键词处理器提取匹配的关键词
    return match_keywords

time_start = time.time()
data_df['Matched_KW'] = data_df.apply(process_keywords, axis=1)
# data_df.to_csv(f'data_df11.csv', index=False)
print(f'关键词提取处理耗时（秒）：' + str(time.time() - time_start))

time_start = time.time()

print(f'关键词提取总数统计耗时（秒）：' + str(time.time() - time_start))

关键词提取处理耗时（秒）：14.701165914535522
关键词提取总数统计耗时（秒）：0.0


In [41]:
data_df

,DOI,KW,AB,TI,YE,CI,Matched_KW
0,10.1287/isre.1100.0313,[HistoryOfIsr],,the early year of iisri recollection of the ed...,2010,"[10.1145/63238.63241, 10.1145/944217.944221, 1...",[]
1,10.1287/isre.1100.0315,"[SocialMedium, SocialCommerce, SponsoredSearch...",SponsoredSearch is the mechanism whereby adver...,SponsoredSearch and MarketEfficiency,2010,"[10.1016/j.jfineco.2004.06.007, 10.1257/aer.99...","[SponsoredSearch, FinancialMarket, SponsoredSe..."
2,10.1287/isre.1080.0218,"[CollaborativeOnlineShopping, SharedNavigation...",prior study investigating businesstoconsumer e...,let shop online together an empirical investig...,2010,"[10.1287/isre.13.4.404.72, 10.1016/S1567-4223(...","[CollaborativeOnlineShopping, CollaborativeOnl..."
3,10.1287/isre.1080.0217,"[KnowledgeManagement, Codification, Knowledge-...",current KnowledgeManagement km technology and ...,the interaction between knowledge Codification...,2010,"[10.2307/3250961, 10.1287/mnsc.49.4.432.14428,...","[KnowledgeManagement, Codification, Codificati..."
4,10.2753/MIS0742-1222270302,"[BusinessProcessOutsourcing, Offshoring, Organ...",this paper identifies and analyzes firmlevel c...,OrganizationalLearning and capability for onsh...,2010,"[10.1002/smj.4250130503, 10.2307/249554, 10.12...","[BusinessProcessOutsourcing, OrganizationalLea..."
...,...,...,...,...,...,...,...
57717,10.17705/1jais.00775,"[InformationPrivacy, PrivacyAAState, PrivacyBo...",to reconcile the personalizationprivacy parado...,reconciling the personalizationprivacy paradox...,2023,"[10.5465/amd.2014.0018, 10.1177/10944281145479...","[PrivacyAAState, PrivacyAAState]"
57718,10.1177/10497323221135974,"[RareDisease, RareDisorder, Disability, Advoca...",in a twostudy project researcher used qualitat...,if not me who Awareness and selfadvocacyrelate...,2023,"[10.1080/13603116.2021.1888323, 10.1111/jopy.1...","[ThematicAnalysis, Awareness, RareDisease, Dis..."
57719,10.1093/jcmc/zmac027,"[SocialPresence, Copresence, Explication, Huma...",initially the province of telecommunication an...,capturing SocialPresence concept Explication t...,2023,"[10.2307/2786027, 10.1162/105474603322761270, ...","[SocialPresence, Explication, SocialPresence, ..."
57720,10.1093/jcmc/zmac024,"[VisualMisinformation, Out-of-contextVisualMis...",a a significant source of misinformation outof...,fighting cheapfakes using a DigitalMediumLiter...,2023,"[10.1080/23299460.2018.1495026, 10.1177/205395...","[OutOfContext, VisualMisinformation, DigitalMe..."


In [5]:
# 03 H词频处理
# 03-1计算驼峰命名后的词频
time_start = time.time()

expanded_df = data_df.explode('KW')
# 在计算词频之前，筛选掉空行和长度小于1的词
expanded_df = expanded_df[
    (expanded_df['KW'].notna()) &  # 不为NaN
    (expanded_df['KW'] != '') &  # 不为空字符串
    (expanded_df['KW'].str.len() >= 1)  # 长度大于等于1
    ]
# 根据'YE'和'KW'进行分组，计算词频
word_frequence_df = expanded_df.groupby(['YE','KW']).size().reset_index(name='Frequency')
# 关键词总数存储
word_frequence_df.to_csv(f'datas\hotness_data\\keyword_Allwordcount.csv', index=False)

print('计算驼峰命名后的词频处理耗时（秒）：' + str(time.time() - time_start))

# 构建词频矩阵
def get_reflection(df):
    unique_keywords = df['KW'].unique()
    unique_years = df['YE'].unique()
    keyword_to_index = {keyword: idx for idx, keyword in enumerate(unique_keywords)}
    year_to_index = {year: idx for idx, year in enumerate(unique_years)}
    # 创建一个空的邻接矩阵
    num_keywords = len(unique_keywords)
    num_years = len(unique_years)
    adjacency_matrix = np.zeros((num_keywords, num_years), dtype=int)
    # 基于数据填充邻接矩阵
    for _, row in df.iterrows():
        idx_keyword = keyword_to_index[row['KW']]
        idx_year = year_to_index[row['YE']]
        adjacency_matrix[idx_keyword, idx_year] += row['Frequency']
    # 存储矩阵结果
    adjacency_matrix_df = pd.DataFrame(adjacency_matrix, index=unique_keywords, columns=unique_years)
    adjacency_matrix_df.to_csv(f'datas\hotness_data\\kewords_count_matrix.csv', index=True)
time_start = time.time()
get_reflection(word_frequence_df)

print('词频矩阵构建耗时（秒）：' + str(time.time() - time_start))

计算驼峰命名后的词频处理耗时（秒）：1.1832292079925537
词频矩阵构建耗时（秒）：12.074662208557129


In [6]:
word_frequence_df

,YE,KW,Frequency
0,2010,'careForGirls'Campaign,1
1,2010,(h)over-bar,1
2,2010,(h)over-bar-core,1
3,2010,2001,1
4,2010,2008PresidentialPrimary,1
...,...,...,...
141888,2023,Zillow,1
141889,2023,Zimbabwe,3
141890,2023,Zoom,1
141891,2023,ZoomElearning,1


In [69]:
# 04 S内容处理(对2019-2022 4年的数据进行处理)
start_year = 2017
# 进行词嵌入
for i in range(4):
    time_start = time.time()
    select_year = start_year + i
    
    # 读取语料库
    corpus = Corpus()
    documents = []
    file_data = data_df[(data_df['YE']>=start_year+i) & (data_df['YE']<=start_year+i+2)]['AB']  # 获取摘要
    for line in file_data.tolist():
        documents.append(line.split())
    corpus.fit(documents)
    
    # 存储摘要单词，为后续有效关键词提取做准备
    # 统计单词出现的次数
    word_counts = Counter(word for sublist in documents for word in sublist)
    with open(f'datas\semantic_data\\word_counts_{start_year+i}_{start_year+i+2}.txt', 'w') as f:
        for word, count in word_counts.items():
            f.write(f"{word} {count}\n")
    
    # 读取待训练数据
    # data_KW_df = data_df['Matched_KW'][(data_df['YE']>=start_year+i) & (data_df['YE']<=start_year+i+2)]
    data_KW_df = data_df['Matched_KW'][(data_df['YE'] == start_year+i+2)]
    all_words = [word for sublist in data_KW_df.values.flatten() for word in sublist if isinstance(sublist, list)]    
    word_list = list(set(all_words))
    
    # 训练模型
    glove = Glove(no_components=50, learning_rate=0.05)
    glove.fit(corpus.matrix, epochs=30, no_threads=4, verbose=True)
    
    # 求出矩阵
    word_vectors_wordlist = glove.word_vectors
    
    # 保存单词矩阵
    with open(f'datas\semantic_data\\Glove_vectors_{start_year+i}_{start_year+i+2}.txt', 'w', encoding='utf-8') as f:
        for word, vector in zip(word_list, word_vectors_wordlist):
            f.write(f"{word} {' '.join(map(str, vector))}\n")
    
    print(f'第{i+1}次处理Glove词嵌入耗时（秒）：' + str(time.time() - time_start))


Performing 30 training epochs with 4 threads
Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7
Epoch 8
Epoch 9
Epoch 10
Epoch 11
Epoch 12
Epoch 13
Epoch 14
Epoch 15
Epoch 16
Epoch 17
Epoch 18
Epoch 19
Epoch 20
Epoch 21
Epoch 22
Epoch 23
Epoch 24
Epoch 25
Epoch 26
Epoch 27
Epoch 28
Epoch 29
第1次处理Glove词嵌入耗时（秒）：83.63347744941711
Performing 30 training epochs with 4 threads
Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7
Epoch 8
Epoch 9
Epoch 10
Epoch 11
Epoch 12
Epoch 13
Epoch 14
Epoch 15
Epoch 16
Epoch 17
Epoch 18
Epoch 19
Epoch 20
Epoch 21
Epoch 22
Epoch 23
Epoch 24
Epoch 25
Epoch 26
Epoch 27
Epoch 28
Epoch 29
第2次处理Glove词嵌入耗时（秒）：85.5649619102478
Performing 30 training epochs with 4 threads
Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7
Epoch 8
Epoch 9
Epoch 10
Epoch 11
Epoch 12
Epoch 13
Epoch 14
Epoch 15
Epoch 16
Epoch 17
Epoch 18
Epoch 19
Epoch 20
Epoch 21
Epoch 22
Epoch 23
Epoch 24
Epoch 25
Epoch 26
Epoch 27
Epoch 28
Epoch 29
第3次处理Glove词嵌

In [9]:
word_frequence_df

,YE,KW,Frequency
0,2010,'careForGirls'Campaign,1
1,2010,(h)over-bar,1
2,2010,(h)over-bar-core,1
3,2010,2001,1
4,2010,2008PresidentialPrimary,1
...,...,...,...
141888,2023,Zillow,1
141889,2023,Zimbabwe,3
141890,2023,Zoom,1
141891,2023,ZoomElearning,1


In [8]:
# 对关键词进行稠密筛选（要对2019-2022四年的词进行处理，3测试1实验）
# 得到年份-关键词集合
keyword_by_year = {}
for item in tqdm(word_frequence_df.values):
    year = item[0]
    keyword = item[1]
    
    if year in keyword_by_year:
        keyword_by_year[year].append(keyword)
    else:
        keyword_by_year[year] = [keyword]
        
for k, v in keyword_by_year.items():
    v = set(v)
# 筛选稠密单词（3年内的）
span = 3
vocab_vector_by_year = {}
for year in range(2017, 2021):
    
    with open(f'datas\semantic_data\\word_counts_{year}_{year+span-1}.txt', 'r') as file:
        vocab = {item.split(' ')[0]: eval(item.split(' ')[1]) for item in file.read().strip().split('\n')}
    
    w_df = word_frequence_df[(word_frequence_df['YE']>=year) & (word_frequence_df['YE']<=year+span-1)]
    w_freq_df = w_df.groupby('KW')['Frequency'].sum().reset_index(name='FrequencyIn3')
    freq_la_3 = set(w_freq_df[w_freq_df['FrequencyIn3']>=3]['KW'].values)
        
    with open(f'datas\semantic_data\\Glove_vectors_{year}_{year+span-1}.txt', 'r') as file:
        vector_set = {}
        
        vec_set = file.read().strip().split('\n')
        for line in tqdm(vec_set):
            v_list = line.split(' ')
            
            word = v_list[0]
            
            # 筛选出现在[19, 20, 21, 22]中，且3年频率大于3的词
            if word in keyword_by_year[year+2] and word in freq_la_3:
                vector = np.array(eval('[' + ','.join(v_list[1:]) + ']'))
                vector_set[word] = vector
        pd.DataFrame(list(vector_set.keys())).to_csv(f'datas\selected_word_{year+2}.txt', index=False)
        print(len(vector_set))
            
    vocab_vector_by_year[year] = (vocab,vector_set)

100%|████████████████████████████████████████████████████████████████████████████| 5726/5726 [00:01<00:00, 5094.54it/s]


1950


100%|████████████████████████████████████████████████████████████████████████████| 6458/6458 [00:01<00:00, 5392.59it/s]


2094


100%|████████████████████████████████████████████████████████████████████████████| 7345/7345 [00:01<00:00, 3865.62it/s]


2449


100%|████████████████████████████████████████████████████████████████████████████| 7752/7752 [00:02<00:00, 3454.32it/s]

2636


In [14]:
for k, v in vocab_vector_by_year.items():
    print(k+2, len(v[1]))

2019 1950
2020 2094
2021 2449
2022 2636


In [16]:
# 05 C引用处理
from collections import defaultdict
from itertools import product

for i in range(2017,2022):
    end_yearnum = 2 + i
    print(f'************{i}——{end_yearnum}年文章引用关系寻找**************')
    new_data_df = data_df[(data_df['YE'] >= i) & (data_df['YE'] <= end_yearnum)]
    
    # 04-1对数据进行处理（把文章一一对应的关系找出来）
    time_start = time.time()  # 记录开始时间
    coneted_data_df = new_data_df.explode('CI')
    DOI_set = set(new_data_df['DOI'])
    CI_list = coneted_data_df['CI']
    print(len(CI_list))
    
    # 使用 isin 方法检查 CR 列中的值是否存在于 CI 列中，并赋值给新列 IsCount
    is_count_list = [1 if CI in DOI_set else '' for CI in CI_list]
    print(len(coneted_data_df))
    print(len(is_count_list))
    coneted_data_df['IsCount'] = is_count_list
    
    coneted_data_df.to_csv(f'datas\citation_data\\article_cite_result_{i}_{end_yearnum}.csv', index=False)
    print('04-1引用关系寻找' + '处理耗时：' + str(time.time() - time_start) + '秒')
    
    # 04-2键词引用网络的excel构建
    def build_keyword_citation(coneted_data_df, data_df):
        DOI_key_list = []
        CI_key_list = []
        # 将 data_df 按照 DOI 列的值进行分组，存储关键词列表字典
        doi_keyword_dict = defaultdict(list)
        for _, row in data_df.iterrows():
            doi_keyword_dict[row['DOI']].extend(row['KW'])
        # 筛选出 IsCount 为 1 的行
        filtered_df = coneted_data_df[coneted_data_df['IsCount'] == 1]
        for _, item in filtered_df.iterrows():
            # 获取当前行的 DOI 和 CR 值
            doi_value = item['DOI']
            ci_value = item['CI']
            # 获取与当前行 DOI 值相同的行的关键词列表
            ci_keywords = doi_keyword_dict.get(ci_value, [])
            # 获取当前行的关键词列表
            doi_keywords = item['KW']
            # 过滤掉非字符串类型的元素或空值
            doi_keywords = [keyword for keyword in doi_keywords if isinstance(keyword, str)]
            ci_keywords = [keyword for keyword in ci_keywords if isinstance(keyword, str)]
            # 使用 product 函数生成笛卡尔积
            for doi_keyword, ci_keyword in product(doi_keywords, ci_keywords):
                DOI_key_list.append(doi_keyword)
                CI_key_list.append(ci_keyword)
        # 构建 DataFrame
        return pd.DataFrame({'DOI_W': DOI_key_list, 'CI_W': CI_key_list})
    
    # 04-2关键词引用网络的excel构建
    time_start = time.time()  
    build_df = build_keyword_citation(coneted_data_df, new_data_df)
    print('04关键词引用网络的excel构建' + '处理耗时：' + str(time.time() - time_start) + '秒')
    
    # 04-3对关键词引用网络进行引用频次计算
    time_start = time.time()  
    # 把关键词引用统计词频
    build_frequency = build_df.groupby(['DOI_W', 'CI_W']).size().reset_index(name='CI_num')
    build_df = pd.merge(build_df,build_frequency, on=['DOI_W','CI_W'], how='left')
    build_df.to_csv(f'datas\citation_data\\keyword_cite_result_{i}_{end_yearnum}.txt', index=False, sep='\t', header=False)
    print('05对关键词引用网络进行引用频次计算' + '处理耗时：' + str(time.time() - time_start) + '秒')

************2017——2019年文章引用关系寻找**************
416093
416093
416093
04-1引用关系寻找处理耗时：13.385972738265991秒
04关键词引用网络的excel构建处理耗时：2.082138776779175秒
05对关键词引用网络进行引用频次计算处理耗时：0.799471378326416秒
************2018——2020年文章引用关系寻找**************
461801
461801
461801
04-1引用关系寻找处理耗时：15.404279708862305秒
04关键词引用网络的excel构建处理耗时：2.2218151092529297秒
05对关键词引用网络进行引用频次计算处理耗时：0.8978545665740967秒
************2019——2021年文章引用关系寻找**************
539371
539371
539371
04-1引用关系寻找处理耗时：18.281686782836914秒
04关键词引用网络的excel构建处理耗时：2.6765294075012207秒
05对关键词引用网络进行引用频次计算处理耗时：1.2092578411102295秒
************2020——2022年文章引用关系寻找**************
604193
604193
604193
04-1引用关系寻找处理耗时：21.033899307250977秒
04关键词引用网络的excel构建处理耗时：2.8435685634613037秒
05对关键词引用网络进行引用频次计算处理耗时：1.2585980892181396秒
************2021——2023年文章引用关系寻找**************
706355
706355
706355
04-1引用关系寻找处理耗时：25.15454411506653秒
04关键词引用网络的excel构建处理耗时：3.3830270767211914秒
05对关键词引用网络进行引用频次计算处理耗时：1.6403276920318604秒


In [17]:
# 最终矩阵（语义内容）—06 关键词语义距离计算及矩阵计算
def get_cos_similar_matrix(M):
    """
    cos_matrix(M) = dot(M, M.T) / dot(norm(M), norm(M).T)
    cos_matrix[i][j] = (M[i] * M[j]) / (norm(M[i]) * norm(M[j])
    """
    
    dot = np.dot(M, M.T)
    norm = np.linalg.norm(M, axis=1).reshape(-1, 1)
    norm_dot = np.dot(norm, norm.T)
    
    return dot / norm_dot

import pickle as pkl

cos_matrix_set = {}
for k, v in tqdm(vocab_vector_by_year.items()):
    vector_set = v[1]
    
    id2word = {}
    matrix = []
    for idx, (w, v) in enumerate(vector_set.items()):
        matrix.append(v)
        id2word[idx] = w
        
    matrix = np.stack(matrix)
    
    
    cos_matrix = get_cos_similar_matrix(matrix)
    
    cos_matrix_set[k] = [id2word, cos_matrix]

100%|████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00, 13.95it/s]


In [102]:
pkl.dump(cos_matrix_set, open(f'datas\semantic_data\\cos_matrix_set.pkl', 'wb'))

In [20]:
cos_matrix_set[2020][1]

array([[ 1.        ,  0.38038698,  0.49498707, ..., -0.48315376,
         0.13556712, -0.57295507],
       [ 0.38038698,  1.        ,  0.33392185, ..., -0.63315399,
         0.04360916, -0.57754014],
       [ 0.49498707,  0.33392185,  1.        , ..., -0.67830193,
        -0.16350788, -0.70846875],
       ...,
       [-0.48315376, -0.63315399, -0.67830193, ...,  1.        ,
         0.04446928,  0.93488822],
       [ 0.13556712,  0.04360916, -0.16350788, ...,  0.04446928,
         1.        ,  0.0317482 ],
       [-0.57295507, -0.57754014, -0.70846875, ...,  0.93488822,
         0.0317482 ,  1.        ]])

In [28]:
# 最终矩阵（语义内容）—06-1 关键词语义内容矩阵存储
for index in range(2017,2021):
    year = index
    time_start = time.time()
    semantic_matrix_df = pd.DataFrame(cos_matrix_set[year][1])
    
    data_list = []
    # 遍历cos_matrix_set[2017-2020][0]中的数据，将每个数据按顺序添加到列表中
    for index, word in sorted(cos_matrix_set[year][0].items()):
        data_list.append(word)
    data_word = pd.DataFrame(data_list, columns=['word'])

    # 合并
    merged_df = pd.concat([data_word, semantic_matrix_df], axis=1)

    merged_df.to_csv(f'datas\prepare_finalmatrix_data\\embedding_distance_{year}_{year+2}.csv', index=False)
    
    print(f'第{year+2}年关键词语义内容矩阵存储耗时：' + str(time.time() - time_start))

第2019年关键词语义内容矩阵存储耗时：5.639269590377808
第2020年关键词语义内容矩阵存储耗时：6.565478563308716
第2021年关键词语义内容矩阵存储耗时：9.106050252914429
第2022年关键词语义内容矩阵存储耗时：10.564378023147583


In [41]:
# 最终矩阵（引用结构）—07 关键词引用过滤计算

# 对关键词引用数据进行过滤，查看2017-2019的是否都在2019之中
def filter_references(input_file, output_file, keys_list):
    with open(input_file, 'r') as f_in, open(output_file, 'w') as f_out:
        for line in f_in:
            source, target, frequency = line.strip().split('\t')
            if source in keys_list and target in keys_list:
                f_out.write(line)
for i in range(2017,2021):
    time_start = time.time()
    
    keys_list = list(vocab_vector_by_year[i][1].keys())
    input_file = f'datas\citation_data\\keyword_cite_result_{i}_{i+2}.txt'
    output_file = f'datas\citation_data\\keyword_cite_result_{i}_{i+2}_filter.txt'
    
    filter_references(input_file, output_file, keys_list)
    
    print(f'第{i+2}年关键词引用数据过滤1耗时：' + str(time.time() - time_start))

for i in range(2017,2021):
    time_start = time.time()
    
    with open(f'datas\citation_data\\keyword_cite_result_{i}_{i+2}_filter.txt', 'r') as file:
        data_chong_lines = file.readlines()
    # 去除重复行
    data_chong_unique_lines = set(data_chong_lines)
    # 将唯一行写入新文件
    with open(f'datas\citation_data\\keyword_cite_result_{i}_{i+2}_filter_unique.txt', 'w') as file:
        file.writelines(data_chong_unique_lines)
        
    print(f'第{i+2}年关键词引用数据过滤2耗时：' + str(time.time() - time_start))
    

第2019年关键词引用数据过滤1耗时：4.726901531219482
第2020年关键词引用数据过滤1耗时：6.0914528369903564
第2021年关键词引用数据过滤1耗时：9.364840745925903
第2022年关键词引用数据过滤1耗时：11.582575798034668
第2019年关键词引用数据过滤2耗时：0.05301213264465332
第2020年关键词引用数据过滤2耗时：0.07703161239624023
第2021年关键词引用数据过滤2耗时：0.09400773048400879
第2022年关键词引用数据过滤2耗时：0.09866809844970703


In [44]:
# 最终矩阵（引用结构）—07-1 关键词引用矩阵计算

for year in range(2017,2021):
    time_start = time.time()
    
    # 对引用结构数据进行处理
    with open(f'datas\selected_word_{year+2}.txt','r') as file:
        word_list = [word.strip() for word in file.readlines()]
    print(len(word_list))
    word_df = pd.DataFrame(0, index=word_list, columns=word_list)

    with open(f'datas\citation_data\\keyword_cite_result_{year}_{year+2}_filter_unique.txt','r') as file:
        wordcite_list = [word.strip().split('\t') for word in file.readlines()]
    print(len(wordcite_list))
    

    # 先查看是否都有
    wordcite_list_new = []
    for line in wordcite_list:
        source, target, frequency = line
        if source in word_list and target in word_list:
            wordcite_list_new.append(line)
    print(len(wordcite_list_new))

    # 对矩阵进行相加
    for row in tqdm(wordcite_list_new):
        citeword, citedword, citenum = row
        word_df.at[citeword, citedword] += int(citenum)
        word_df.at[citedword, citeword] += int(citenum)  # 更新对称位置的值

    # 对角线上的值减去自身
    for i in tqdm(range(len(word_df)), desc="Diagonal Processing"):
        word_df.iloc[i, i] -= word_df.iloc[i, i]
    # print(word_df)
    word_df.to_csv(f'datas\prepare_finalmatrix_data\\keyword_cite_matrix_{year}_{year+2}-{year+3}.csv')
    
    print(f'第{year+2}年关键词引用矩阵构建耗时：' + str(time.time() - time_start))

1950
40418
40418


Diagonal Processing: 100%|███████████████████████████████████████████████████████| 1950/1950 [00:00<00:00, 6188.95it/s]


第2019年关键词引用矩阵构建耗时：2.95603346824646
2094
49640
49640


Diagonal Processing: 100%|███████████████████████████████████████████████████████| 2094/2094 [00:00<00:00, 5864.24it/s]


第2020年关键词引用矩阵构建耗时：3.8749020099639893
2449
65200
65200


Diagonal Processing: 100%|███████████████████████████████████████████████████████| 2449/2449 [00:00<00:00, 6035.68it/s]


第2021年关键词引用矩阵构建耗时：4.898732423782349
2636
74210
74210


Diagonal Processing: 100%|███████████████████████████████████████████████████████| 2636/2636 [00:00<00:00, 6025.96it/s]


第2022年关键词引用矩阵构建耗时：5.728400230407715


In [46]:
# 最终矩阵（词频数量）—08-1 关键词词频矩阵计算

# 读取 CSV 文件
data = pd.read_csv(f'datas\hotness_data\\kewords_count_matrix.csv')

# 删除指定的列
data1 = data.drop(columns=['2021', '2022', '2023'])
data2 = data.drop(columns=['2010', '2022', '2023'])
data3 = data.drop(columns=['2010', '2011', '2023'])
data4 = data.drop(columns=['2010', '2011', '2012'])

# 将处理后的数据保存回 CSV 文件
data1.to_csv('datas\hotness_data\\kewords_count_matrix_2010-2019.csv', index=False)
data2.to_csv('datas\hotness_data\\kewords_count_matrix_2011-2020.csv', index=False)
data3.to_csv('datas\hotness_data\\kewords_count_matrix_2012-2021.csv', index=False)
data4.to_csv('datas\hotness_data\\kewords_count_matrix_2013-2022.csv', index=False)

In [49]:
# 最终矩阵（词频数量）—08-1 关键词词频矩阵计算

# 读取总词频参考表
total_word_freq = pd.read_csv('datas\hotness_data\\kewords_count_matrix.csv')

# 读取H原始矩阵数据，删除不在单词列表中的列
for i in range(2019,2023):
    time_start = time.time()
    # 读取原始矩阵数据
    original_matrix = pd.read_csv(f'datas\hotness_data\\kewords_count_matrix_{i-9}-{i}.csv')
    original_matrix_df = pd.DataFrame(original_matrix)
    original_matrix_df = original_matrix_df.rename(columns={'Unnamed: 0': 'word'})

    # 读取单词列表
    with open(f'datas\selected_word_{i}.txt', 'r') as f:
        word_order = [word.strip() for word in f.readlines()]

    # 删除不在单词列表中的列,行判断并筛选数据
    filtered_df = original_matrix_df[original_matrix_df['word'].isin(word_order)]
    
    # 对矩阵进行排序
    sorted_matrix = filtered_df.set_index('word').loc[word_order].reset_index()

    sorted_matrix.to_csv(f'datas\prepare_finalmatrix_data\\kewords_count_matrix_{i-9}-{i}_filter_{i+1}.csv', index=False)
    
    print(f'第{i}年关键词词频矩阵构建耗时：' + str(time.time() - time_start))


第2019年关键词词频矩阵构建耗时：0.08773231506347656
第2020年关键词词频矩阵构建耗时：0.10402154922485352
第2021年关键词词频矩阵构建耗时：0.12204146385192871
第2022年关键词词频矩阵构建耗时：0.0959160327911377
